# IMports 

In [2]:
import pandas as pd 
import numpy as np
from sentence_transformers import SentenceTransformer
import mlflow 
import time 
from sklearn.metrics.pairwise import cosine_similarity 

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# LOad data

In [3]:
df = pd.read_csv('../data/complaints_clean.csv')
df_sample = df.sample(1000, random_state=42).reset_index(drop=True)
df_sample.shape

(1000, 4)

In [4]:
test_queries = [
    ("credit card fraud unauthorized charges", "Credit card"),
    ("mortgage foreclosure loan modification", "Mortgage"),
    ("debt collector harassment phone calls", "Debt collection"),
    ("student loan repayment servicer", "Student loan"),
    ("bank account unauthorized transfer", "Bank account or service"),
]

# Evaluation function

In [11]:
def evaluate_model(model_name, df_sample, test_queries):
    print(f"\nLoading {model_name}...")
    model = SentenceTransformer(model_name)
    
    # Encode corpus
    start = time.time()
    embeddings = model.encode(
        df_sample['clean_narrative'].tolist(),
        batch_size=64,
        show_progress_bar=True
    )
    encoding_time = time.time() - start
    throughput = len(df_sample) / encoding_time
    
    # Evaluate top-1 accuracy on test queries
    correct = 0
    scores = []
    
    for query, expected_product in test_queries:
        query_emb = model.encode([query])
        sims = cosine_similarity(query_emb, embeddings)[0]
        top_idx = sims.argmax()
        top_score = sims[top_idx]
        top_product = df_sample.iloc[top_idx]['product']
        
        if top_product == expected_product:
            correct += 1
        scores.append(top_score)
    
    top1_accuracy = correct / len(test_queries)
    avg_score = np.mean(scores)
    
    return {
        "model": model_name,
        "encoding_time_seconds": round(encoding_time, 2),
        "throughput_per_second": round(throughput, 2),
        "embedding_dim": embeddings.shape[1],
        "top1_accuracy": round(top1_accuracy, 4),
        "avg_similarity_score": round(avg_score, 4)
    }

# MLFLow

In [17]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("Semantic-search-model-comparison")

/opt/miniconda3/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/15 09:12:35 INFO mlflow.tracking.fluent: Experiment with name 'Semantic-search-model-comparison' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/spectra/Desktop/ml-ai-journey/Week-8/Project-2-Semantic-Search/notebooks/../mlruns/477950621532171937', creation_time=1776224555236, experiment_id='477950621532171937', last_update_time=1776224555236, lifecycle_stage='active', name='Semantic-search-model-comparison', tags={}, trace_location=None, workspace='default'>

In [18]:
model_to_compare = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2"
]

In [19]:
results = []

In [20]:
for model_name in model_to_compare:
    with mlflow.start_run(run_name=model_name):
        metrics = evaluate_model(model_name,df_sample,test_queries)

        mlflow.log_param("model_name", metrics["model"])
        mlflow.log_param("embedding_dim", metrics["embedding_dim"])
        mlflow.log_metric("encoding_time_seconds", metrics["encoding_time_seconds"])
        mlflow.log_metric("throughput_per_second", metrics["throughput_per_second"])
        mlflow.log_metric("top1_accuracy", metrics["top1_accuracy"])
        mlflow.log_metric("avg_similarity_score", metrics["avg_similarity_score"])

        results.append(metrics)
        print(f"\n{model_name} done")


Loading all-MiniLM-L6-v2...


Loading weights: 100%|████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4181.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|████████████████████████████████████████████████████████████████| 16/16 [00:24<00:00,  1.53s/it]



all-MiniLM-L6-v2 done

Loading all-mpnet-base-v2...


Loading weights: 100%|████████████████████████████████████████████████████| 199/199 [00:00<00:00, 7281.08it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|████████████████████████████████████████████████████████████████| 16/16 [02:56<00:00, 11.04s/it]



all-mpnet-base-v2 done


In [21]:
results_df = pd.DataFrame(results)
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))


=== Model Comparison ===
            model  encoding_time_seconds  throughput_per_second  embedding_dim  top1_accuracy  avg_similarity_score
 all-MiniLM-L6-v2                  24.47                  40.87            384            1.0                0.6416
all-mpnet-base-v2                 176.71                   5.66            768            1.0                0.6536
